In [1]:
# load_faq_data: FAQ 데이터 로드 함수 가져오기
from ingest import load_faq_data
# load_faq_data(): 전체 데이터를 메모리로 불러오는 함수
documents = load_faq_data()

In [2]:
documents[10]

{'id': 'f3dd94f323',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'WSL2: ResponseError: model requires more system memory (X.X GiB) than is available (Y.Y GiB). My system has more than X.X GiB.',
 'answer': 'Your WSL2 is set to use Y.Y GiB, not all your computer memory. To allocate more RAM, follow these steps:\n\n1. Create a `.wslconfig` file under your Windows user profile directory: `C:\\Users\\YourUsername\\.wslconfig`.\n\n2. Include the desired RAM allocation in the file:\n\n   ```ini\n   [wsl2]\n   memory=8GB\n   ```\n\n3. Restart WSL using the command:\n\n   ```bash\n   wsl --shutdown\n   ```\n\n4. Run the `free` command in WSL to verify the changes.\n\nFor more details, read [this article](https://www.aleksandrhovhannisyan.com/blog/limiting-memory-usage-in-wsl-2/).'}

In [3]:
# documents_llm: llm-zoomcamp 과정 문서만 담을 리스트 초기화
documents_llm = []
# for doc: 전체 문서를 순회하며 데이터 필터링
for doc in documents:
    # if doc["course"]: 'llm-zoomcamp' 과정명과 일치하는지 확인하는 조건문
    if doc["course"] == "llm-zoomcamp":
        # .append(): 조건에 맞는 문서를 리스트에 추가
        documents_llm.append(doc)
# len(): 필터링된 문서 총 개수 확인
len(documents_llm)
# 이제부터 이 문서들만 사용할 것이므로 documents로 지정하겠습니다.

79

In [4]:
# documents: 전체 데이터셋을 필터링된 문서셋으로 갱신
documents = documents_llm
#각 문서는 이미 id 필드를 가지고 있습니다:

In [5]:
# doc: 샘플로 첫 번째 문서 가져오기
doc = documents[0]
# .id: 문서 고유 식별자 출력
print(doc["id"])
# .question: 원본 질문 출력
print(doc["question"])
# .answer: 원본 답변 출력
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [6]:
# Pydantic을 활용한 구조화된 출력 원리 설명 코드
# BaseModel: LLM에게 "이러한 JSON 구조로만 응답하라"는 데이터 계약서(Schema)를 정의함
from pydantic import BaseModel

# Questions: LLM의 응답을 파이썬 객체로 변환하기 위한 규격 클래스
# 1. 모델 정의: LLM에게 출력할 데이터의 '약속된 양식(Schema)'을 미리 규격화함.
#    LLM은 이 클래스 구조를 보고, 결과물을 텍스트가 아닌 JSON 데이터로 구성함.

class Questions(BaseModel):
    # 2. 타입 강제: 'questions' 필드는 반드시 문자열 리스트(list[str])여야 함을 정의.
    #    만약 LLM이 이 형식을 지키지 않으면 Pydantic이 파싱 에러를 발생시켜 데이터 무결성을 보장함.
# questions: 이 필드는 LLM이 JSON의 'questions' 키에 리스트 형태로 데이터를 담게 유도함
    # Pydantic이 이 구조를 기반으로 LLM의 응답 텍스트를 파싱하여 즉시 파이썬 list[str] 객체로 변환해줌[cite: 1]
    questions: list[str]

# [원리 요약]
# API 호출 시 text_format=Questions를 넘기면:
# 1) Pydantic 모델이 'JSON Schema'로 변환되어 LLM에게 전달됨.
# 2) LLM은 약속된 JSON 양식에 맞춰 답변을 생성함.
# 3) OpenAI 응답 파서(openai_client.responses.parse())가 해당 JSON을 파이썬 객체(Questions 인스턴스)로 자동 변환함.
# 4) 덕분에 result.questions 와 같이 리스트 객체처럼 즉시 접근 가능해짐.

In [7]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [8]:
# OpenAI API 통신을 위한 표준 라이브러리인 OpenAI 클래스를 가져옵니다.
from openai import OpenAI
# 환경 변수 파일을 로드하기 위한 dotenv 관련 모듈을 가져옵니다.
from dotenv import load_dotenv

# .env 파일에 저장된 환경 변수를 시스템 환경 변수로 불러옵니다.
load_dotenv()

# Ollama가 돌아가고 있는 주소로 직접 연결합니다.
OLLAMA_BASE_URL = "http://localhost:11434/v1"

# Ollama 서버와 통신하기 위한 OpenAI 클라이언트 인스턴스를 생성합니다.
client = OpenAI(
    base_url=OLLAMA_BASE_URL, 
    api_key="ollama"  # Ollama는 로컬이므로 아무 문자열이나 넣어도 됩니다.
)

LLM_MODEL = "qwen2.5:0.5b" 

# 이미 클라이언트가 정의되어 있다면, 바로 위의 모델 호출 코드를 다시 실행해 보세요.

# 연결 설정을 마치고 정상적으로 완료되었음을 출력합니다.
print("클라이언트 연결 완료")

클라이언트 연결 완료


In [9]:
# json.dumps(): 파이썬 객체인 문서를 LLM이 이해 가능한 직렬화된 JSON 문자열로 변환

# doc 객체를 LLM이 프롬프트로 이해할 수 있도록 JSON 문자열 형식으로 변환합니다.
import json

# json.dumps()를 사용하여 딕셔너리 구조를 LLM 입력용 텍스트로 직렬화합니다.
user_prompt = json.dumps(doc)

In [10]:
# 변환된 프롬프트가 올바르게 구성되었는지 확인합니다.
user_prompt

'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [11]:
# 1. 메시지 구성
# messages: 모델에게 시스템 지침과 참조할 문맥 데이터를 체계적으로 전달

# LLM에게 전달할 메시지 리스트를 구성합니다.
# "developer" 역할은 LLM의 행동 양식(질문 생성 규칙)을 정의합니다.
# "user" 역할은 우리가 방금 변환한 FAQ 문서(user_prompt)를 입력값으로 전달합니다.
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

# 구성된 메시지 객체를 확인하여, 올바른 역할과 내용이 할당되었는지 체크합니다.
messages

In [13]:
# 텍스트에서 질문 리스트를 추출하기 위한 정규 표현식 모듈을 가져옵니다.
import re

# LLM의 응답 텍스트를 입력받아 5개의 질문 리스트로 변환하는 함수입니다.
def extract_questions_from_text(text):
    # 1. '1. 질문'과 같이 번호가 붙은 형식에서 질문 텍스트만 추출합니다.
    questions = re.findall(r"\d+\.\s*(.*)", text)
    
    # 2. 만약 번호 형식이 없다면, 줄바꿈(\n)을 기준으로 텍스트를 나누어 처리합니다.
    if not questions:
        questions = [line.strip() for line in text.split("\n") if line.strip()]
        
    # 3. 최대 5개의 질문만 반환하도록 제한합니다.
    return questions[:5]

# 함수가 정상적으로 메모리에 로드되었음을 알립니다.
print("질문 추출 함수 등록 완료")


함수 등록 완료


In [14]:
# response.output_parsed.questions

# 2. Ollama 모델 호출

# Ollama 모델을 호출하여 답변(질문 리스트)을 생성합니다.
# temperature=0.0으로 설정하여 매번 일관된 답변을 하도록 유도합니다.
response = client.chat.completions.create(
    model=LLM_MODEL,	# 'qwen2.5:0.5b'로 설정했던 그 변수입니다.
    messages=messages,
    temperature=0.0
)

# 1. 모델 응답에서 텍스트만 추출
# 모델이 생성한 전체 응답 내용 중 본문(content)만 추출합니다.
# 만약 응답이 비어있을 경우를 대비해 빈 문자열을 기본값으로 설정합니다.
content = response.choices[0].message.content or ""

# 2. 이전에 정의해둔 추출 함수를 사용하여 리스트로 변환
# (이 함수는 앞서 정의하신 extract_questions_from_text를 그대로 사용합니다.)
questions_list = extract_questions_from_text(content)

# 3. 이제 이 questions_list를 사용하세요
print("추출된 질문 리스트:", questions_list)

추출된 질문 리스트: ["**Course Participation**: You can continue to participate in the course as long as you haven't received a certificate yet.", '**Certificate Requirements**: To receive a certificate, you need to submit your project along with any additional materials or information required by the course organizers.', 'Go back to the course page where you registered for the course.', 'Click on "Submit Project" if it\'s still available.', 'If there is an option to submit your project, do so along with any additional materials or information required by the course organizers.']


In [15]:
doc

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [16]:
# Ollama 사용: llm_structured 미사용


In [18]:
# 현재 모델 호출 후 발생한 토큰 사용량을 저장할 변수를 선언합니다.
# usage

In [19]:
# 비용 계산을 위한 유틸리티 함수를 불러옵니다.
# from evaluation_utils import calc_price

In [20]:
# 로컬(Ollama) 환경에서는 비용이 발생하지 않으므로, 이 코드는 참조용으로만 둡니다.
# calc_price(usage)

{'input_cost': 0.0, 'output_cost': 0.0, 'total_cost': 0.0}

In [40]:
# 결과를 담을 빈 리스트를 초기화합니다.
records = []

# 앞서 추출한 질문 리스트(questions_list)를 순회하며 개별 레코드를 구성합니다.
for q in questions_list:
    # 질문 내용과 현재 문서의 고유 ID를 딕셔너리로 묶어 리스트에 추가합니다.
    records.append({
        "question": q,
        "document": doc["id"]
    })

# 구성이 완료된 레코드 리스트를 출력하여 확인합니다.
print(records)

[{'question': "**Course Participation**: You can continue to participate in the course as long as you haven't received a certificate yet.", 'document': '74eb249bbf'}, {'question': '**Certificate Requirements**: To receive a certificate, you need to submit your project along with any additional materials or information required by the course organizers.', 'document': '74eb249bbf'}, {'question': 'Go back to the course page where you registered for the course.', 'document': '74eb249bbf'}, {'question': 'Click on "Submit Project" if it\'s still available.', 'document': '74eb249bbf'}, {'question': 'If there is an option to submit your project, do so along with any additional materials or information required by the course organizers.', 'document': '74eb249bbf'}]


In [41]:
import pandas as pd

In [42]:
pd.DataFrame(records)

,question,document
0,**Course Participation**: You can continue to ...,74eb249bbf
1,**Certificate Requirements**: To receive a cer...,74eb249bbf
2,Go back to the course page where you registere...,74eb249bbf
3,"Click on ""Submit Project"" if it's still availa...",74eb249bbf
4,"If there is an option to submit your project, ...",74eb249bbf


In [24]:
# Ollama 사용: llm_structured_retry 미사용


In [43]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "developer", "content": data_gen_instructions},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.0,
    )

    content = response.choices[0].message.content or ""
    questions_list = extract_questions_from_text(content)

    results = []
    for q in questions_list:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, None

In [45]:
# 현재 doc을 대상으로 함수를 실행하여 결과를 확인합니다.
results, usage = generate_ground_truth(doc)
results

[{'question': "**Course Participation**: You can continue to participate in the course as long as you haven't received a certificate yet.",
  'document': '74eb249bbf'},
 {'question': '**Certificate Requirements**: To receive a certificate, you need to submit your project along with any additional materials or information required by the course organizers.',
  'document': '74eb249bbf'},
 {'question': 'Go back to the course page where you registered for the course.',
  'document': '74eb249bbf'},
 {'question': 'Click on "Submit Project" if it\'s still available.',
  'document': '74eb249bbf'},
 {'question': 'If there is an option to submit your project, do so along with any additional materials or information required by the course organizers.',
  'document': '74eb249bbf'}]

In [27]:
# [과거 테스트용 샘플링 루프]
# 전체 79개 문서 루프를 돌리기 전, 5개의 샘플 문서만 추출하여 테스트하던 로직입니다.
# 현재는 CSV 파일을 통한 전체 데이터 로드로 프로세스가 변경되었습니다.

# from tqdm.auto import tqdm # 루프 진행 상황을 시각적으로 보여주는 프로그레스 바 라이브러리

# ground_truth = [] # 결과값을 담을 리스트 초기화
# usages = []       # 사용량 데이터를 담을 리스트 초기화

# for doc in tqdm(documents[:5]): # 전체 리스트에서 앞의 5개 문서만 골라 순회
#     records, usage = generate_ground_truth(doc) # 해당 문서에 대한 질문 생성 실행
#     ground_truth.extend(records)                # 생성된 질문들을 결과 리스트에 누적. 
                                                  # 리스트 끝에 다른 리스트의 모든 요소를 추가하여 하나로 병합합니다.
                                                  # (append는 리스트 자체를 하나의 객체로 넣지만, extend는 내부 요소들을 개별적으로 풀어 넣기 위해 사용합니다.)
#     usages.append(usage)                        # 토큰 사용량 정보 기록


In [28]:
# [과거 병렬 처리 로직]
# 79개의 문서에 대해 API를 병렬 호출하여 정답 데이터를 생성하던 과거의 방식입니다.
# 현재는 CSV 파일을 직접 로드하므로 사용하지 않습니다.

# from concurrent.futures import ThreadPoolExecutor  # 스레드 기반 병렬 작업을 위한 라이브러리
# from evaluation_utils import map_progress          # 병렬 작업의 진행 상황을 모니터링하기 위한 유틸리티

# with ThreadPoolExecutor(max_workers=6) as pool:    # 6개의 워커 스레드를 사용하여 데이터 생성 속도 향상
#     results = map_progress(pool, documents, generate_ground_truth) # 전체 문서에 대해 질문 생성 함수 적용

In [46]:
# [주석 처리] 루프를 통한 데이터 취합 단계 생략

# [데이터 취합 루프 생략]
# 아래 루프는 과거 API를 통해 정답 데이터를 직접 생성할 때 사용되던 방식입니다.
# 현재는 구축된 CSV 파일을 로드하므로, 별도의 병렬 처리 및 취합 과정이 불필요하여 주석 처리합니다.
# ground_truth = [] # 정답 데이터를 담을 리스트
# usages = []       # API 호출 시 사용된 토큰량을 담을 리스트
# for records, usage in results: # 루프를 통해 생성된 결과값들을 순차적으로 취합
#     ground_truth.extend(records) # 리스트 끝에 다른 리스트의 모든 요소를 추가하여 하나로 병합합니다.
                                   # (append는 리스트 자체를 하나의 객체로 넣지만, extend는 내부 요소들을 개별적으로 풀어 넣기 위해 사용합니다.)
#     usages.append(usage)         # 사용량을 리스트에 추가

# 1. 데이터 처리를 위해 pandas 라이브러리를 임포트합니다.
import pandas as pd

# 2. 정답 데이터셋(CSV)이 저장된 로컬 경로를 지정합니다.
file_path = r"E:\IT_SPACES\AI\ZoomCamp\LLM\04\2026\Evaluation\data\ground-truth-data.csv"

# 3. CSV 파일을 읽어와 데이터프레임(df) 형태로 변환합니다.
# read_csv를 통해 원본 데이터를 파이썬에서 다루기 쉬운 행렬 구조로 올립니다.
df_ground_truth = pd.read_csv(file_path)

# 4. 분석 및 루프 처리가 용이하도록 데이터프레임을 리스트 내 딕셔너리 구조로 변환합니다.
# orient="records" 옵션은 각 행을 {"question": "...", "document": "..."} 형태의 딕셔너리로 만듭니다.
ground_truth = df_ground_truth.to_dict(orient="records")

# 5. 최종적으로 로드된 레코드의 총 개수를 출력하여 데이터 확보 상태를 확인합니다.
print(f"데이터 로드 완료: {len(ground_truth)}개의 정답 레코드 확보")

데이터 로드 완료: 395개의 정답 레코드 확보


In [32]:
ground_truth[10]

{'question': 'I just found this course. Am I still allowed to join now, and does that affect my chance to get a certificate?',
 'course': 'llm-zoomcamp',
 'document': '74eb249bbf'}

In [34]:
# [주석 처리] usages 리스트를 직접 생성하지 않았으므로 이 계산 단계는 건너뜁니다.
# from evaluation_utils import calc_price
# total_cost = 0.0
# for usage in usages:
#     cost = calc_price(usage)
#     total_cost = total_cost + cost["total_cost"]
# total_cost

print("비용 계산 단계는 CSV 로드 방식이므로 건너뜁니다.")

비용 계산 단계는 CSV 로드 방식이므로 건너뜁니다.


In [47]:
# df_ground_truth.to_csv("data/ground_truth.csv", index=False)
import pandas as pd

# 1. 데이터프레임 생성
df_ground_truth = pd.DataFrame(ground_truth)

# 2. 지정된 폴더에 저장 (이미 파일이 있다면 덮어씁니다)
save_path = r"E:\IT_SPACES\AI\ZoomCamp\LLM\04\2026\Evaluation\data\ground-truth.csv"
df_ground_truth.to_csv(save_path, index=False)

print(f"데이터가 {save_path}에 성공적으로 저장되었습니다.")

데이터가 E:\IT_SPACES\AI\ZoomCamp\LLM\04\2026\Evaluation\data\ground-truth.csv에 성공적으로 저장되었습니다.


In [48]:
len(df_ground_truth)

395